<a href="https://colab.research.google.com/github/Swathi04705/Applied_Linear_Algebra/blob/main/ALA4_Glove.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import numpy as np
import gzip
import urllib.request
import os

# Download the exact GloVe model required by Assignment 2
MODEL_URL = (
    "https://github.com/piskvorky/gensim-data/releases/download/"
    "glove-wiki-gigaword-50/glove-wiki-gigaword-50.gz"
)

MODEL_FILE = "/content/glove-wiki-gigaword-50.gz"

if not os.path.exists(MODEL_FILE):
    print("Downloading GloVe model...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_FILE)

print("Download complete!")


# Lightweight model class without Gensim
class GloveModel:

    def __init__(self):
        self.vector_size = 50
        self.key_to_index = {}
        self.vectors = None

    def __getitem__(self, word):
        if word not in self.key_to_index:
            raise KeyError(word)

        return self.vectors[self.key_to_index[word]]


model = GloveModel()

# Read the downloaded compressed model
print("Loading word vectors...")

words = []
vectors = []

with gzip.open(MODEL_FILE, "rt", encoding="utf-8") as file:

    first_line = True

    for line in file:

        parts = line.rstrip().split()

        if first_line:
            first_line = False

            # Skip word2vec header if present
            if len(parts) == 2:
                try:
                    int(parts[0])
                    int(parts[1])
                    continue
                except ValueError:
                    pass

        if len(parts) != 51:
            continue

        word = parts[0]
        vector = np.asarray(parts[1:], dtype=np.float32)

        model.key_to_index[word] = len(words)
        words.append(word)
        vectors.append(vector)

model.vectors = np.vstack(vectors)

print("GloVe model loaded successfully!")
print("Vocabulary size:", len(model.key_to_index))
print("Vector dimension:", model.vector_size)
print("Example vector shape:", model["research"].shape)


Download complete!
Loading word vectors...
GloVe model loaded successfully!
Vocabulary size: 400000
Vector dimension: 50
Example vector shape: (50,)


In [2]:
tags = [
    "research",
    "innovation",
    "education",
    "university",
    "students",
    "faculty",
    "campus",
    "engineering",
    "medicine",
    "technology",
    "curriculum",
    "collaboration",
    "publication",
    "laboratory",
    "scholarship",
    "mentorship",
    "internship",
    "entrepreneurship",
    "accreditation",
    "alumni"
]

assert len(tags) == 20, "Exactly 20 tags are required."

missing_tags = [tag for tag in tags if tag not in model.key_to_index]
if missing_tags:
    raise ValueError(f"These tags are missing from the GloVe vocabulary: {missing_tags}")

print("Number of tags:", len(tags))
print(tags)


Number of tags: 20
['research', 'innovation', 'education', 'university', 'students', 'faculty', 'campus', 'engineering', 'medicine', 'technology', 'curriculum', 'collaboration', 'publication', 'laboratory', 'scholarship', 'mentorship', 'internship', 'entrepreneurship', 'accreditation', 'alumni']


In [3]:
# TEMPORARY TEST TEXT ONLY.
# Replace this with up to 400 words copied from real Manipal University LinkedIn post(s).
INPUT_TEXT = """
Manipal University brings together students faculty and researchers through education
innovation engineering technology research and collaboration. The university campus
supports laboratory activities internships scholarship mentorship and entrepreneurship.
Students participate in academic programmes conferences publications and professional
development initiatives. Faculty members guide research projects and encourage innovation
through collaboration with industry and academic partners.
"""

# Add the exact URL for every LinkedIn post used.
# Example format: SOURCE_CITATIONS = ["https://www.linkedin.com/posts/..."]
SOURCE_CITATIONS = [
    "REPLACE_WITH_THE_REAL_LINKEDIN_POST_URL"
]

print("Input word count:", len(INPUT_TEXT.split()))
print("Source URLs entered:", SOURCE_CITATIONS)


Input word count: 53
Source URLs entered: ['REPLACE_WITH_THE_REAL_LINKEDIN_POST_URL']


In [5]:
import string
STOPWORDS = {
    "the", "a", "an", "and", "or", "of", "to", "in", "on", "for",
    "is", "are", "was", "were", "with", "at", "by", "from",
    "it", "this", "that", "these", "those", "our", "we", "us",
    "their", "they", "them", "its", "as", "be", "been", "being",
    "through", "into", "which", "who", "what", "when", "where",
    "how", "can", "will", "has", "have", "had"
}

def preprocess_text(text, model, stopwords):
    original_tokens = text.split()
    lower_text = text.lower()
    no_punctuation = lower_text.translate(str.maketrans("", "", string.punctuation))
    tokens = no_punctuation.split()
    after_stopwords = [word for word in tokens if word not in stopwords]
    filtered_tokens = [word for word in after_stopwords if len(word) > 2]

    in_vocab_tokens = [word for word in filtered_tokens if word in model.key_to_index]
    oov_tokens = [word for word in filtered_tokens if word not in model.key_to_index]

    stats = {
        "before_preprocessing": len(original_tokens),
        "after_preprocessing": len(filtered_tokens),
        "in_vocabulary": len(in_vocab_tokens),
        "out_of_vocabulary": len(oov_tokens),
        "distinct_oov": sorted(set(oov_tokens))
    }
    return filtered_tokens, in_vocab_tokens, oov_tokens, stats

processed_tokens, in_vocab_tokens, oov_tokens, token_stats = preprocess_text(
    INPUT_TEXT, model, STOPWORDS
)

print("TOKEN STATISTICS")
for key, value in token_stats.items():
    print(f"{key}: {value}")

if len(in_vocab_tokens) == 0:
    raise ValueError("No usable in-vocabulary words remain after preprocessing.")


TOKEN STATISTICS
before_preprocessing: 53
after_preprocessing: 42
in_vocabulary: 42
out_of_vocabulary: 0
distinct_oov: []


In [6]:
def build_tag_matrix(model, tags):
    if len(tags) != 20:
        raise ValueError("The tag vocabulary must contain exactly 20 tags.")

    rows = []
    tag_names = []

    for tag in tags:
        components = tag.lower().split()
        missing = [word for word in components if word not in model.key_to_index]
        if missing:
            raise ValueError(f"Tag '{tag}' has out-of-vocabulary component(s): {missing}")

        component_vectors = np.array([model[word] for word in components], dtype=float)
        tag_vector = np.mean(component_vectors, axis=0)
        rows.append(tag_vector)
        tag_names.append(tag)

    return tag_names, np.vstack(rows)


def build_text_matrix(model, tokens):
    in_vocab_tokens = []
    oov_tokens = []
    rows = []

    for token in tokens:
        if token in model.key_to_index:
            in_vocab_tokens.append(token)
            rows.append(model[token])
        else:
            oov_tokens.append(token)

    if rows:
        W = np.vstack(rows).astype(float)
    else:
        W = np.empty((0, model.vector_size), dtype=float)

    return in_vocab_tokens, oov_tokens, W


tag_names, T = build_tag_matrix(model, tags)
in_vocab_tokens, oov_tokens, W = build_text_matrix(model, processed_tokens)

print("T.shape =", T.shape)
print("W.shape =", W.shape)

assert T.shape == (20, 50)
assert W.shape == (len(in_vocab_tokens), 50)
assert W.shape[0] > 0


T.shape = (20, 50)
W.shape = (42, 50)


In [7]:
class Vector:
    def __init__(self, values):
        self.values = tuple(float(x) for x in values)

    def add(self, other):
        if len(self.values) != len(other.values):
            raise ValueError("Vectors must have the same dimension.")
        return Vector(a + b for a, b in zip(self.values, other.values))

    def scale(self, scalar):
        return Vector(scalar * x for x in self.values)

    def dot(self, other):
        if len(self.values) != len(other.values):
            raise ValueError("Vectors must have the same dimension.")
        return sum(a * b for a, b in zip(self.values, other.values))

    def norm(self):
        return float(np.sqrt(sum(x * x for x in self.values)))

    def cosine_similarity(self, other):
        denominator = self.norm() * other.norm()
        if denominator == 0:
            raise ValueError("Cosine similarity is undefined for a zero vector.")
        return self.dot(other) / denominator


def similarity_matrix_vector_class(W, T):
    n_words = W.shape[0]
    n_tags = T.shape[0]
    S = []

    for i in range(n_words):
        row = []
        word_vector = Vector(W[i])
        for j in range(n_tags):
            tag_vector = Vector(T[j])
            row.append(word_vector.cosine_similarity(tag_vector))
        S.append(row)

    return np.array(S, dtype=float)


S_vector = similarity_matrix_vector_class(W, T)
print("S_vector.shape =", S_vector.shape)
assert S_vector.shape == (W.shape[0], 20)


S_vector.shape = (42, 20)


In [8]:
def similarity_matrix_numpy(W, T):
    W_norm = np.linalg.norm(W, axis=1, keepdims=True)
    T_norm = np.linalg.norm(T, axis=1, keepdims=True)

    if np.any(W_norm == 0):
        raise ValueError("W contains a zero vector; row normalization is undefined.")
    if np.any(T_norm == 0):
        raise ValueError("T contains a zero vector; row normalization is undefined.")

    W_hat = W / W_norm
    T_hat = T / T_norm

    return W_hat @ T_hat.T


S_numpy = similarity_matrix_numpy(W, T)
print("S_numpy.shape =", S_numpy.shape)
assert S_numpy.shape == (W.shape[0], 20)


S_numpy.shape = (42, 20)


In [9]:
same_results = np.allclose(S_vector, S_numpy, rtol=1e-5, atol=1e-7)
max_difference = np.max(np.abs(S_vector - S_numpy))

print("np.allclose(S_vector, S_numpy):", same_results)
print("Maximum absolute difference:", max_difference)

assert same_results, "The Vector-class and NumPy similarity matrices do not agree."

# Check one word/tag pair independently using the Vector class.
word_index = 0
tag_index = 0
single_pair_score = Vector(W[word_index]).cosine_similarity(Vector(T[tag_index]))

print(
    f"Example pair: word='{in_vocab_tokens[word_index]}', "
    f"tag='{tag_names[tag_index]}'"
)
print("Vector-class pair score:", single_pair_score)
print("Corresponding NumPy matrix entry:", S_numpy[word_index, tag_index])
assert np.isclose(single_pair_score, S_numpy[word_index, tag_index], atol=1e-7)


np.allclose(S_vector, S_numpy): True
Maximum absolute difference: 4.440892098500626e-16
Example pair: word='manipal', tag='research'
Vector-class pair score: 0.1942422095322948
Corresponding NumPy matrix entry: 0.1942422095322948


In [10]:
def rank_tags(tag_names, S, in_vocab_tokens):
    if S.shape != (len(in_vocab_tokens), len(tag_names)):
        raise ValueError("Similarity matrix dimensions do not match the tokens and tags.")

    rankings = []
    for j, tag in enumerate(tag_names):
        best_word_index = int(np.argmax(S[:, j]))
        best_score = float(S[best_word_index, j])
        best_word = in_vocab_tokens[best_word_index]

        rankings.append({
            "tag": tag,
            "score": best_score,
            "best_matching_word": best_word,
            "identical_match": (best_word == tag)
        })

    rankings.sort(key=lambda item: item["score"], reverse=True)
    return rankings


rankings = rank_tags(tag_names, S_numpy, in_vocab_tokens)

print(f"{'Rank':<6}{'Tag':<20}{'Score':<14}{'Best-matching word':<24}{'Match type'}")
print("-" * 85)

for rank, item in enumerate(rankings[:8], start=1):
    match_type = "Identical" if item["identical_match"] else "Different word"
    print(
        f"{rank:<6}{item['tag']:<20}{item['score']:<14.6f}"
        f"{item['best_matching_word']:<24}{match_type}"
    )


Rank  Tag                 Score         Best-matching word      Match type
-------------------------------------------------------------------------------------
1     research            1.000000      research                Identical
2     innovation          1.000000      innovation              Identical
3     university          1.000000      university              Identical
4     engineering         1.000000      engineering             Identical
5     scholarship         1.000000      scholarship             Identical
6     entrepreneurship    1.000000      entrepreneurship        Identical
7     education           1.000000      education               Identical
8     students            1.000000      students                Identical


In [11]:
def run_unit_tests():
    # 1. Tag matrix shape
    assert T.shape == (20, 50)

    # 2. Text matrix shape
    assert W.shape == (len(in_vocab_tokens), 50)

    # 3. Similarity matrix shape
    assert S_numpy.shape == (len(in_vocab_tokens), 20)

    # 4. One pair agrees between Vector and NumPy
    assert np.isclose(
        Vector(W[0]).cosine_similarity(Vector(T[0])),
        S_numpy[0, 0],
        atol=1e-7
    )

    # 5. Both complete matrices agree
    assert np.allclose(S_vector, S_numpy, rtol=1e-5, atol=1e-7)

    # 6. OOV text token is excluded and reported
    test_tokens = ["research", "zzzxxyy_oov_token"]
    test_in_vocab, test_oov, test_W = build_text_matrix(model, test_tokens)
    assert "zzzxxyy_oov_token" in test_oov
    assert "zzzxxyy_oov_token" not in test_in_vocab
    assert test_W.shape == (len(test_in_vocab), 50)

    # 7. OOV tag component must raise an error
    bad_tags = tags[:-1] + ["zzzxxyy_oov_token"]
    error_raised = False
    try:
        build_tag_matrix(model, bad_tags)
    except ValueError:
        error_raised = True
    assert error_raised, "An OOV tag component should raise ValueError."

    print("All 7 required unit-test checks passed.")


run_unit_tests()


All 7 required unit-test checks passed.
